<a href="https://colab.research.google.com/github/OjasTamhankar/AML-Lab/blob/main/exp08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# # Load your downloaded Amazon CSV
# df_amazon = pd.read_csv('/content/drive/MyDrive/Datasets/Reviews.csv')   # Change path if needed

In [2]:
# print("Original Shape:", df_amazon.shape)
# print(df_amazon.columns.tolist())

In [3]:
# df_amazon = df_amazon[['Text', 'Summary']].copy()

In [4]:
#df_amazon.rename(columns={'Text': 'text', 'Summary': 'summary'}, inplace=True)

In [5]:
# Basic cleaning
# df_amazon = df_amazon.dropna().reset_index(drop=True)   # Remove rows with missing values

# # Remove very short or empty summaries
# df_amazon = df_amazon[df_amazon['summary'].str.len() > 5]
# df_amazon = df_amazon[df_amazon['text'].str.len() > 50]

# print("After cleaning Amazon shape:", df_amazon.shape)
# df_amazon.head()

In [6]:
!pip install datasets -q

from datasets import load_dataset

# Load SAMSum dataset
dataset = load_dataset("knkarthick/samsum")

# Convert to pandas DataFrame (we'll use train split)
df_samsum = pd.DataFrame({
    'text': dataset['train']['dialogue'],
    'summary': dataset['train']['summary']
})

print("SAMSum Shape:", df_samsum.shape)
df_samsum.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


SAMSum Shape: (14731, 2)


,text,summary
0,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."


In [7]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)          # Remove extra spaces
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)  # Keep only useful characters
    return text.strip()

# # Clean Amazon
# df_amazon['text'] = df_amazon['text'].apply(clean_text)
# df_amazon['summary'] = df_amazon['summary'].apply(clean_text)

# Clean SAMSum
df_samsum['text'] = df_samsum['text'].apply(clean_text)
df_samsum['summary'] = df_samsum['summary'].apply(clean_text)

# Remove duplicates
# df_amazon = df_amazon.drop_duplicates()
df_samsum = df_samsum.drop_duplicates()

In [8]:
# Take a manageable sample from Amazon (it's very large)
# df_amazon_sample = df_amazon.sample(n=15000, random_state=42)   # You can change 15000

# Combine Amazon sample + all of SAMSum
df_combined = df_samsum

print("Final Combined Dataset Shape:", df_combined.shape)

# Shuffle the data
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# Optional: Save the combined dataset
df_combined.to_csv('/content/combined_summarization_data.csv', index=False)

Final Combined Dataset Shape: (14730, 2)


In [9]:
# Add special tokens to summary
df_combined['summary'] = df_combined['summary'].apply(lambda x: 'sostok ' + str(x) + ' eostok')

In [10]:
df_combined = df_combined.sample(n=14000, random_state=42)

In [11]:
! pip install tensorflow

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define max lengths
max_text_len = 100
max_summary_len = 50

# Tokenizer for text (encoder)
x_tokenizer = Tokenizer(num_words=10000)
x_tokenizer.fit_on_texts(df_combined['text'])

x_vocab_size = len(x_tokenizer.word_index) + 1
print("Text Vocabulary Size:", x_vocab_size)

# Tokenizer for summary (decoder)
y_tokenizer = Tokenizer(num_words=5000)
y_tokenizer.fit_on_texts(df_combined['summary'])

y_vocab_size = len(y_tokenizer.word_index) + 1
print("Summary Vocabulary Size:", y_vocab_size)

Text Vocabulary Size: 30038
Summary Vocabulary Size: 17158


In [13]:
# Convert text to sequences
encoder_input = x_tokenizer.texts_to_sequences(df_combined['text'])
encoder_input = pad_sequences(encoder_input, maxlen=max_text_len, padding='post')

# Convert summary to sequences
decoder_input = y_tokenizer.texts_to_sequences(df_combined['summary'])
decoder_input = pad_sequences(decoder_input, maxlen=max_summary_len, padding='post')

In [14]:
import numpy as np

decoder_target = np.zeros_like(decoder_input)

for i in range(len(decoder_input)):
    decoder_target[i, :-1] = decoder_input[i, 1:]

In [15]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

latent_dim = 256
embedding_dim = 128

# -------- Encoder --------
encoder_inputs = Input(shape=(max_text_len,))
enc_emb = Embedding(x_vocab_size, embedding_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_output, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# -------- Decoder --------
decoder_inputs = Input(shape=(max_summary_len,))
dec_emb = Embedding(y_vocab_size, embedding_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(y_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# -------- Model --------
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='rmsprop',
    loss='sparse_categorical_crossentropy'
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 128)  │  3,844,864 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 128)   │  2,196,224 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    394,240 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 50, 256), │    394,240 │ embedding_1[0][0… │
│                     │ (None, 256),      │            │ lstm[0][1],       │
│                     │ (None, 256)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 17158) │  4,409,606 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,239,174 (42.87 MB)

 Trainable params: 11,239,174 (42.87 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=64,
    epochs=15,   # increase from 5 → 15
    validation_split=0.1
)

print(history.history['loss'])
print(history.history['val_loss'])

model.save('/content/summarization_model.h5')

from google.colab import drive
drive.mount('/content/drive')

import pickle

# Save model
model.save('/content/drive/MyDrive/summarization_model.h5')

# Save tokenizers
with open('/content/drive/MyDrive/x_tokenizer.pkl', 'wb') as f:
    pickle.dump(x_tokenizer, f)

with open('/content/drive/MyDrive/y_tokenizer.pkl', 'wb') as f:
    pickle.dump(y_tokenizer, f)

print("All files saved to Google Drive")

Epoch 1/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 26s 80ms/step - loss: 3.1067 - val_loss: 2.5195
Epoch 2/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 80ms/step - loss: 2.5333 - val_loss: 2.4874
Epoch 3/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 2.4968 - val_loss: 2.4311
Epoch 4/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 83ms/step - loss: 2.4322 - val_loss: 2.3793
Epoch 5/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 2.3822 - val_loss: 2.3341
Epoch 6/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 17s 84ms/step - loss: 2.3399 - val_loss: 2.2966
Epoch 7/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 2.3040 - val_loss: 2.2644
Epoch 8/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 2.2754 - val_loss: 2.2348
Epoch 9/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 2.2370 - val_loss: 2.2078
Epoch 10/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 2.2114 - val_loss: 2.1893
Epoch 11/15
197/197 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 2.1888 - val_loss: 2.1703
Epoch 12/15
197/197 ━━━━━━━━━━

[3.1066901683807373, 2.5332887172698975, 2.4968457221984863, 2.4322128295898438, 2.382205009460449, 2.3399291038513184, 2.3039815425872803, 2.2753634452819824, 2.2369725704193115, 2.2113583087921143, 2.188832998275757, 2.1701714992523193, 2.1526784896850586, 2.137742519378662, 2.131835699081421]
[2.51946759223938, 2.4873979091644287, 2.431051731109619, 2.379305362701416, 2.334123134613037, 2.296616792678833, 2.2643988132476807, 2.234752655029297, 2.2078278064727783, 2.1893324851989746, 2.170250177383423, 2.153714656829834, 2.1400232315063477, 2.1275100708007812, 2.116713285446167]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All files saved to Google Drive


In [22]:
from tensorflow.keras.models import load_model
import pickle

# Load model
model = load_model('/content/summarization_model.h5')

# Load tokenizers
with open('/content/drive/MyDrive/x_tokenizer.pkl', 'rb') as f:
    x_tokenizer = pickle.load(f)

with open('/content/drive/MyDrive/y_tokenizer.pkl', 'rb') as f:
    y_tokenizer = pickle.load(f)

max_text_len = 100
max_summary_len = 50
latent_dim = 256

In [23]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

# Encoder
encoder_inputs = model.input[0]
encoder_emb = model.layers[2].output
encoder_lstm = model.layers[4]

encoder_outputs, state_h, state_c = encoder_lstm(encoder_emb)
encoder_model = Model(encoder_inputs, [state_h, state_c])

# Decoder
decoder_inputs = model.input[1]

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_emb_layer = model.layers[3]
decoder_lstm = model.layers[5]
decoder_dense = model.layers[6]

decoder_emb = decoder_emb_layer(decoder_inputs)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_emb,
    initial_state=decoder_states_inputs
)

decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs, state_h, state_c]
)

In [24]:
reverse_target_word_index = y_tokenizer.index_word
target_word_index = y_tokenizer.word_index

In [25]:
def decode_sequence(input_text):

    seq = x_tokenizer.texts_to_sequences([input_text])
    seq = pad_sequences(seq, maxlen=max_text_len, padding='post')

    states_value = encoder_model.predict(seq, verbose=0)

    target_seq = np.zeros((1,1))
    target_seq[0,0] = target_word_index['sostok']

    decoded_sentence = ''

    for _ in range(max_summary_len):

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == 'eostok':
            break

        decoded_sentence += ' ' + sampled_word

        target_seq[0,0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence.strip()

In [26]:
text = "The product quality is very good and battery lasts long."

print("Input:", text)
print("Summary:", decode_sequence(text))

Input: The product quality is very good and battery lasts long.
Summary: kate will be in the party


In [27]:
text = "The product quality is very good and battery lasts long."
print("Input:", text)
print("Summary:", decode_sequence(text))

Input: The product quality is very good and battery lasts long.
Summary: kate will be in the party


In [28]:
text = "Let's meet tomorrow at 5 pm near the college gate."

print("Input:", text)
print("Summary:", decode_sequence(text))

Input: Let's meet tomorrow at 5 pm near the college gate.
Summary: kate will be in the party
